# sum-back-expand-broadcast — worked example 1: Sum Backward — keepdim=True, broadcast to original

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `sum-back-expand-broadcast`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When you sum a tensor along an axis with `keepdim=True`, the output already has a size-1 placeholder at the summed dimension. The backward pass simply needs to broadcast `grad_out` back to the original input shape using `.expand(x.shape)`. No unsqueeze is needed because the axis was never removed.

## Worked solution

**Step 1 — Understand the shapes.**
If `x` has shape `(3, 4)` and we call `x.sum(dim=1, keepdim=True)`, the output has shape `(3, 1)`. The size-1 axis at position 1 is already there.

**Step 2 — Why expand works.**
Each output entry `out[i, 0]` was formed by summing all four entries in `x[i, :]`. The local derivative of each `x[i, j]` with respect to `out[i, 0]` is 1. So `grad_in[i, j] = grad_out[i, 0]` for every j — the same gradient value replicates across all four columns.

**Step 3 — Implement.**
Since `grad_out` has shape `(3, 1)` and `x.shape` is `(3, 4)`, we simply call `grad_out.expand(x.shape)`. PyTorch broadcasting rules expand any size-1 dimension along columns.

In [ ]:
import torch as t

def sum_back_keepdim(grad_out: t.Tensor, x: t.Tensor, dim: int) -> t.Tensor:
    """Backward for out = x.sum(dim=dim, keepdim=True)."""
    # keepdim=True means grad_out already has size-1 at dim; just expand.
    return grad_out.expand(x.shape)

# Demonstrate
t.manual_seed(42)
x = t.randn(3, 4)
out = x.sum(dim=1, keepdim=True)          # shape (3, 1)
grad_out = t.ones_like(out)               # upstream gradient, shape (3, 1)
grad_in = sum_back_keepdim(grad_out, x, dim=1)
print('x.shape:', x.shape)               # (3, 4)
print('grad_out.shape:', grad_out.shape) # (3, 1)
print('grad_in.shape:', grad_in.shape)   # (3, 4)
print('grad_in:\n', grad_in)             # all ones — every entry contributed equally